# WeatherWhisper API Demo (Colab)

A simple, standalone demo: get a free [OpenWeatherMap](https://openweathermap.org/api) API key working and call it to fetch live weather data for a city.

The first part is intentionally just the raw API call — no LLM involved. The goal is to see a real request succeed with your own key, and understand the response shape you get back. At the end, we combine it with [Groq](https://console.groq.com/) to build a tiny weather-summary project.

### 1. Install `requests`

Colab already ships `requests`, but this keeps the demo self-contained in case you're running on a fresh runtime that doesn't have it.

In [ ]:
!pip install -q requests

### 2. Get your free OpenWeather API key

1. Go to https://openweathermap.org/api
2. Click **Sign Up** (top right) and create a free account (email + password, or sign up with Google).
3. Verify your email if prompted.
4. Once logged in, go to https://home.openweathermap.org/api_keys — a **default key is auto-generated** for every new account, so there's no separate "create key" step needed for the free tier.
5. Copy that key — it's what you'll paste in as `OPENWEATHER_API_KEY` in the next cell.
6. **Important gotcha:** a freshly created key can take up to ~10–15 minutes to activate. If you get a `401 Unauthorized` error right after signing up, it almost always means the key isn't active *yet* — not that it's wrong. Wait a few minutes and retry before assuming something's broken.
7. The "Current Weather Data" endpoint (the one this demo uses) is included in the free tier by default — no upgrade needed. You can confirm this under https://home.openweathermap.org/subscriptions.

### 3. Set your OpenWeather API key

Uses Colab's `userdata` secrets (padlock icon in the left sidebar) if you've set one there named `OPENWEATHER_API_KEY` — so you don't paste your key in plaintext into a cell. Falls back to an interactive prompt if the secret isn't set.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENWEATHER_API_KEY"] = userdata.get("OPENWEATHER_API_KEY")
except Exception:
    import getpass
    os.environ["OPENWEATHER_API_KEY"] = getpass.getpass("Enter your OPENWEATHER_API_KEY: ")

assert os.environ.get("OPENWEATHER_API_KEY"), "OPENWEATHER_API_KEY was not set"

### 4. Call the Current Weather API

The endpoint is `https://api.openweathermap.org/data/2.5/weather`, taking a city name (`q`), your key (`appid`), and a units system (`units`) as query params. `units=metric` returns temperatures in Celsius — use `units=imperial` for Fahrenheit, or omit `units` entirely to get Kelvin (the API's raw default).

In [ ]:
import json

import requests

WEATHER_URL = "https://api.openweathermap.org/data/2.5/weather"


def get_weather(city: str) -> dict:
    """Fetches the current weather for a city. Raises a clear error on
    a non-200 response (bad city name, inactive/invalid key, rate limit)
    instead of a confusing raw KeyError from the caller trying to read
    a field that was never returned."""
    response = requests.get(
        WEATHER_URL,
        params={
            "q": city,
            "appid": os.environ["OPENWEATHER_API_KEY"],
            "units": "metric",
        },
    )
    if response.status_code != 200:
        raise RuntimeError(
            f"OpenWeather API returned {response.status_code}: {response.json().get('message', response.text)}\n"
            "If you just created your API key, it may still be activating (can take up to ~10-15 minutes)."
        )
    return response.json()


data = get_weather("London")
print(json.dumps(data, indent=2))

### 5. Pull out the human-relevant fields

The raw response has a lot in it — here's how you'd actually use it: read the specific fields you care about out of that same `data` dict.

In [ ]:
city_name = data["name"]
temp = data["main"]["temp"]
feels_like = data["main"]["feels_like"]
humidity = data["main"]["humidity"]
description = data["weather"][0]["description"]

print(f"{city_name}: {temp}°C (feels like {feels_like}°C), {description}, humidity {humidity}%")

### 6. Try it yourself

Change `city` below and re-run the cell to look up a different location.

In [ ]:
city = "Tokyo"

data = get_weather(city)
print(json.dumps(data, indent=2))

### 7. Notes / caveats (OpenWeather)

- **Free-tier rate limits**: 60 calls/minute, 1,000 calls/day. Plenty for a live demo, but don't loop over hundreds of cities in a tight loop without a delay.
- **New keys take time to activate** — up to ~10-15 minutes after signup. A `401` right after creating your key is expected, not a bug; just wait and retry.
- **`units=metric` is why `temp`/`feels_like` come back in Celsius.** Use `units=imperial` for Fahrenheit, or drop the param entirely for Kelvin (the API's raw default) — a common gotcha if you copy this code without noticing that param.
- **City name ambiguity**: `q="Springfield"` will resolve to *some* Springfield, not necessarily the one you meant. For unambiguous results, OpenWeatherMap also supports lookup by geographic coordinates (`lat`/`lon` params) or a `city,country_code` pair (e.g. `q="London,GB"`).

---

## Mini project: weather + Groq

Now let's combine the raw weather data above with an LLM call, so it can turn the JSON into a natural-language summary — a first taste of "grounding" a model with live, real-world data instead of just its training knowledge.

### 8. Get your free Groq API key

1. Go to https://console.groq.com/keys
2. Sign up / log in (email, Google, or GitHub).
3. Click **Create API Key**, give it any name, and copy the key — it's only shown once.
4. Groq's free tier is generous and needs no payment method to get started, but it does have rate limits per model (requests/minute and tokens/minute) — if you see a `429` error, you've hit one; wait a bit and retry.

### 9. Install the Groq SDK and set your key

Same secret-handling pattern as the OpenWeather key above: Colab `userdata` first, falling back to an interactive prompt.

In [ ]:
!pip install -q groq

In [ ]:
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

assert os.environ.get("GROQ_API_KEY"), "GROQ_API_KEY was not set"

### 10. Weather + Groq: a plain-English weather summary

Fetch the weather for a city (reusing `get_weather()` from earlier), then hand the raw JSON to a Groq-hosted model and ask it to summarize what a traveler would actually want to know.

In [ ]:
from groq import Groq

groq_client = Groq()


def summarize_weather(city: str) -> str:
    weather_data = get_weather(city)

    prompt = f"""You are a helpful travel assistant. Given this raw weather API response for {city}, write a 2-3 sentence plain-English summary a traveler packing for today would find useful (temperature, how it feels, and anything notable like rain/wind/humidity).

Weather data:
{json.dumps(weather_data)}
"""

    completion = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_completion_tokens=200,
    )
    return completion.choices[0].message.content


print(summarize_weather("Paris"))

### 11. Try it yourself

Change the city below and re-run — same `get_weather()` call as before, now grounding Groq's summary in that live data.

In [ ]:
print(summarize_weather("Mumbai"))

### 12. Notes / caveats (Groq mini project)

- **Why this matters**: the LLM never "knows" today's weather from training — it only knows what's in the prompt. This is the simplest possible example of grounding a model with live, external data instead of relying on (possibly stale or absent) training knowledge.
- **Model name may change over time.** `openai/gpt-oss-20b` is current as of writing — check https://console.groq.com/docs/models for the latest available models if this one is ever deprecated.
- **Rate limits**: Groq's free tier limits both requests/minute and tokens/minute per model. A `429` error means you've hit one — space out calls or switch to a smaller/faster model if you're calling this in a loop.

In [1]:
data = {
  "coord": {
    "lon": -0.1257,
    "lat": 51.5085
  },
  "weather": [
    {
      "id": 801,
      "main": "Clouds",
      "description": "few clouds",
      "icon": "02d"
    }
  ],
  "base": "stations",
  "main": {
    "temp": 13.89,
    "feels_like": 13.29,
    "temp_min": 13.89,
    "temp_max": 13.89,
    "pressure": 1031,
    "humidity": 75,
    "sea_level": 1031,
    "grnd_level": 1029
  },
  "visibility": 10000,
  "wind": {
    "speed": 2.35,
    "deg": 82,
    "gust": 6.09
  },
  "clouds": {
    "all": 14
  },
  "dt": 1793450730,
  "sys": {
    "country": "GB",
    "sunrise": 1793429508,
    "sunset": 1793464577
  },
  "timezone": 0,
  "id": 2643743,
  "name": "London",
  "cod": 200
}


In [2]:
type(data)

dict

In [ ]:
ChatCompletion(
    id="chatcmpl-5a4af97d-8d87-4a98-9273-b97abf1d1efe",

    choices=[
        Choice(
            finish_reason="length",
            index=0,
            logprobs=None,

            message=ChatCompletionMessage(
                content="It’s a mild day in Paris with the temperature hovering around 14°C (about 13°C feels-like). The sky is clear, humidity is moderate at 67%, and a light",

                role="assistant",
                annotations=None,
                executed_tools=None,
                function_call=None,

                reasoning="We need to produce 2-3 sentence plain-English summary..."
                tool_calls=None
            )
        )
    ],

    created=1793451475,
    model="openai/gpt-oss-20b",
    object="chat.completion",

    usage=CompletionUsage(
        completion_tokens=200,
        prompt_tokens=358,
        total_tokens=558,

        completion_time=0.203906655,
        prompt_time=0.008101812,
        total_time=0.212008467
    ),

    service_tier="on_demand",
    system_fingerprint="fp_ed70b23331"
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (427588129.py, line 18)

In [9]:
completion = {
    "id": "chatcmpl-12345",
    "object": "chat.completion",
    "created": 1793523131,
    "model": "openai/gpt-oss-20b",
    "choices": [
        {
            "finish_reason": "stop",
            "index": 0,
            "message": {
                "content": "It is a cool day in Paris with a temperature of 8°C.",
                "role": "assistant",
                "reasoning": "The temperature is 8°C with clear skies.",
                "tool_calls": None
            }
        }
    ],
    "usage": {
        "prompt_tokens": 364,
        "completion_tokens": 200,
        "total_tokens": 564
    }
}

'It is a cool day in Paris with a temperature of 8°C.'